In [1]:
#from IPython.core.display import HTML
#HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [2]:
import pandas as pd
import numpy as np
import warnings

from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.utils import resample
from sklearn.model_selection import StratifiedKFold, cross_validate

warnings.filterwarnings('ignore')


In [3]:
class RotationForestClassifier(ClassifierMixin, BaseEstimator):
    
    # Etiqueta absoluta para que Scikit-Learn JAMÁS lo trate como regresor
    _estimator_type = "classifier"
    
    def __init__(self, n_estimators=50, max_features_per_subset=15, random_state=None):
        self.n_estimators = n_estimators
        self.max_features_per_subset = max_features_per_subset
        self.random_state = random_state
        
    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.models = []
        self.rotation_matrices = []
        # Definición obligatoria para clasificadores en scikit-learn
        self.classes_ = np.unique(y) 
        
        n_features = X.shape[1]
        
        for i in range(self.n_estimators):
            features = np.random.permutation(n_features)
            subsets = [features[j:j + self.max_features_per_subset] 
                       for j in range(0, n_features, self.max_features_per_subset)]
            
            rotation_matrix = np.zeros((n_features, n_features))
            
            for subset in subsets:
                X_subset = X[:, subset]
                # Remuestreo (Bootstrap)
                X_sample = resample(X_subset, n_samples=int(X.shape[0]*0.75), random_state=self.random_state) 
                
                pca = PCA()
                pca.fit(X_sample)
                
                for j, feature_idx in enumerate(subset):
                    rotation_matrix[feature_idx, subset] = pca.components_[:, j]
                    
            X_rotated = np.dot(X, rotation_matrix)
            
            tree = DecisionTreeClassifier(random_state=self.random_state)
            tree.fit(X_rotated, y)
            
            self.models.append(tree)
            self.rotation_matrices.append(rotation_matrix)
            
        return self
    
    def predict_proba(self, X):
        probas = np.zeros((X.shape[0], len(self.classes_)))
        for tree, rot_mat in zip(self.models, self.rotation_matrices):
            X_rotated = np.dot(X, rot_mat)
            probas += tree.predict_proba(X_rotated)
        return probas / self.n_estimators
        
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

    # Método de seguridad para validadores de scikit-learn 1.4+
    def _more_tags(self):
        return {"estimator_type": "classifier"}


In [4]:

# 2. Carga y preprocesamiento

df = pd.read_csv('/kaggle/input/datasets/nico14h/metagenomica/dataset.csv', sep='\t')

df_pivot = df.pivot(index='Paciente', columns='Bacteria_Limpia', values='AbundanciaTaxonomica')
df_pivot = df_pivot.fillna(0) 

labels = df[['Paciente', 'Diagnosticos']].drop_duplicates().set_index('Paciente')
data = df_pivot.join(labels)

X = data.drop(columns=['Diagnosticos']).values
y = data['Diagnosticos'].values

# 3. Entrenamiento y Evaluación
rf = RotationForestClassifier(n_estimators=50, max_features_per_subset=15, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Ejecutamos el cross validation con n_jobs=1 para evitar errores de paralelización ocultos
results = cross_validate(rf, X, y, cv=cv, scoring=['accuracy', 'f1', 'roc_auc'], n_jobs=1)

# 4. Resultados finales
print("--- MÉTRICAS DEL ROTATION FOREST ---")
print(f"Accuracy : {np.mean(results['test_accuracy']):.4f} ± {np.std(results['test_accuracy']):.4f}")
print(f"F1 Score : {np.mean(results['test_f1']):.4f} ± {np.std(results['test_f1']):.4f}")
print(f"ROC AUC  : {np.mean(results['test_roc_auc']):.4f} ± {np.std(results['test_roc_auc']):.4f}")

--- MÉTRICAS DEL ROTATION FOREST ---
Accuracy : 0.7943 ± 0.0749
F1 Score : 0.8364 ± 0.0554
ROC AUC  : 0.9123 ± 0.0469
